### Kronos로 시장 예측

In [1]:
# 라이브러리
import sys
import json
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime, timedelta

sys.path.append('/Users/honghyeongi/policonomy/third_party/Kronos')

from model import Kronos, KronosTokenizer, KronosPredictor
import torch

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'디바이스: {device}')

디바이스: mps


In [2]:
# Kronos 모델로드
tokenizer = KronosTokenizer.from_pretrained("NeoQuasar/kronos-Tokenizer-base")
model = Kronos.from_pretrained("NeoQuasar/kronos-small")
predictor = KronosPredictor(model, tokenizer, max_context=512)

config.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/15.8M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/228 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/99.0M [00:00<?, ?B/s]

In [3]:
# 시장 데이터 수집
def get_ohlcv(ticker: str, period: str = '6mo') -> pd.DataFrame:
    '''yfinance로 OHLCV 데이터 수집'''
    stock = yf.Ticker(ticker)
    df = stock.history(period=period)
    df = df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
    df.columns = ['open', 'high', 'low', 'close', 'volume']
    df.index = pd.to_datetime(df.index)
    df = df.dropna()
    print(f'{ticker} 데이터 수집 완료: {len(df)}개')
    return df

# 주요 지수/종목 수집
targets = {
    'kospi':  '^KS11',   # 코스피
    'usd_krw': 'KRW=X',  # 달러/원
    'samsung': '005930.KS',  # 삼성전자
    'nasdaq':  '^IXIC',  # 나스닥
}

market_data = {}
for name, ticker in targets.items():
    market_data[name] = get_ohlcv(ticker)

^KS11 데이터 수집 완료: 124개
KRW=X 데이터 수집 완료: 130개
005930.KS 데이터 수집 완료: 124개
^IXIC 데이터 수집 완료: 124개


In [10]:
# Kronos 예측
def predict_market(name: str, df: pd.DataFrame, pred_days: int = 5) -> dict:
    lookback = min(400, len(df) - pred_days)

    x_df = df.iloc[:lookback][['open', 'high', 'low', 'close', 'volume']].reset_index(drop=True)
    x_ts = pd.Series(pd.to_datetime(df.index[:lookback]).tz_localize(None) if df.index.tz is not None else df.index[:lookback])
    y_ts = pd.Series(pd.to_datetime(df.index[lookback:lookback + pred_days]).tz_localize(None) if df.index.tz is not None else df.index[lookback:lookback + pred_days])

    try:
        pred_df = predictor.predict(
            df=x_df,
            x_timestamp=x_ts,
            y_timestamp=y_ts,
            pred_len=pred_days,
            T=1.0,
            top_p=0.9,
            sample_count=3
        )

        last_close = float(df['close'].iloc[lookback - 1])
        pred_close = float(pred_df['close'].iloc[-1])
        change_pct = (pred_close - last_close) / last_close * 100

        result = {
            'name': name,
            'last_close': round(last_close, 2),
            'pred_close': round(pred_close, 2),
            'change_pct': round(change_pct, 2),
            'direction': '상승' if change_pct > 0 else '하락',
            'pred_range': {
                'high': round(float(pred_df['high'].max()), 2),
                'low': round(float(pred_df['low'].min()), 2),
            }
        }
        print(f'{name}: {result["direction"]} ({change_pct:+.2f}%)')
        return result

    except Exception as e:
        print(f'{name} 예측 실패 - {e}')
        return None

test = predict_market('kospi', market_data['kospi'])
print(json.dumps(test, ensure_ascii=False, indent=2))

100%|██████████| 5/5 [00:00<00:00, 31.63it/s]

kospi: 하락 (-2.07%)
{
  "name": "kospi",
  "last_close": 5377.3,
  "pred_close": 5265.9,
  "change_pct": -2.07,
  "direction": "하락",
  "pred_range": {
    "high": 5450.85,
    "low": 5193.47
  }
}


In [11]:
# 전체 예측
predictions = {}
for name, df in market_data.items():
    result = predict_market(name, df)
    if result:
        predictions[name] = result
    
print(json.dumps(predictions, ensure_ascii=False, indent=2))

100%|██████████| 5/5 [00:00<00:00, 32.93it/s]


kospi: 하락 (-3.50%)


100%|██████████| 5/5 [00:01<00:00,  3.90it/s]


usd_krw: 하락 (-0.79%)


100%|██████████| 5/5 [00:00<00:00, 28.28it/s]

samsung: 하락 (-1.50%)



100%|██████████| 5/5 [00:00<00:00, 34.27it/s]

nasdaq: 상승 (+0.55%)
{
  "kospi": {
    "name": "kospi",
    "last_close": 5377.3,
    "pred_close": 5189.29,
    "change_pct": -3.5,
    "direction": "하락",
    "pred_range": {
      "high": 5376.65,
      "low": 5064.81
    }
  },
  "usd_krw": {
    "name": "usd_krw",
    "last_close": 1509.26,
    "pred_close": 1497.26,
    "change_pct": -0.79,
    "direction": "하락",
    "pred_range": {
      "high": 1517.59,
      "low": 1496.18
    }
  },
  "samsung": {
    "name": "samsung",
    "last_close": 186200.0,
    "pred_close": 183404.12,
    "change_pct": -1.5,
    "direction": "하락",
    "pred_range": {
      "high": 194932.59,
      "low": 180282.19
    }
  },
  "nasdaq": {
    "name": "nasdaq",
    "last_close": 21879.18,
    "pred_close": 21999.0,
    "change_pct": 0.55,
    "direction": "상승",
    "pred_range": {
      "high": 22291.33,
      "low": 21602.96
    }
  }
}


In [21]:
# 정성 분석과 융합
from openai import OpenAI

ollama = OpenAI(
    base_url='http://localhost:11434/v1',
    api_key='ollama'
)
MODEL = 'gemma4:e4b'

def fuse_predictions(predictions: dict, analyzed_articles: list) -> dict:
    top_issues = [a['issue'] for a in analyzed_articles[:5]]
    issues_text = '\n'.join(f'- {issue}' for issue in top_issues)

    kronos_summary = '\n'.join([
        f'- {v["name"]}: {v["direction"]} ({v["change_pct"]:+.2f}%)'
        for v in predictions.values() if v
    ])

    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[{
            'role': 'user',
            'content': f'''Respond in JSON only. No markdown. No explanation.

Market predictions:
{kronos_summary}

Top news:
{issues_text}

JSON format:
{{"kospi": {{"direction": "하락", "reason": "reason", "confidence": 70}}, "usd_krw": {{"direction": "약세", "reason": "reason", "confidence": 65}}, "key_sector": {{"name": "반도체", "direction": "강세", "reason": "reason"}}, "overall_sentiment": "negative", "weekly_summary": "summary"}}'''
        }],
        max_tokens=2048,
        temperature=0.1
    )

    text = response.choices[0].message.content.strip()
    lines = [l for l in text.split('\n') if not l.strip().startswith('```')]
    text = '\n'.join(lines).strip()

    start_idx = text.find('{')
    end_idx = text.rfind('}') + 1
    if end_idx > 0:
        try:
            result = json.loads(text[start_idx:end_idx])
            return result
        except Exception as e:
            print(f'파싱 실패: {e}')
    return {}

# 분석 기사 로드
with open('../data/analyzed_articles.json', 'r', encoding='utf-8') as f:
    analyzed = json.load(f)

fused = fuse_predictions(predictions, analyzed)
print(json.dumps(fused, ensure_ascii=False, indent=2))

{
  "kospi": {
    "direction": "하락",
    "reason": "중동발 에너지 위기 및 기업의 자산 오용 문제 등 거시적 리스크가 시장 전반의 하락 압력으로 작용함.",
    "confidence": 75
  },
  "usd_krw": {
    "direction": "약세",
    "reason": "글로벌 불확실성 속에서 안전자산 선호 심리가 일부 작용하며 원화 가치가 상대적으로 강세를 보일 수 있음.",
    "confidence": 65
  },
  "key_sector": {
    "name": "반도체",
    "direction": "강세",
    "reason": "글로벌 수요 회복 기대와 주요 기업들의 핵심 기술 집중 전략(구조조정)이 섹터의 내재적 강세를 뒷받침함.",
    "confidence": 70
  },
  "overall_sentiment": "negative",
  "weekly_summary": "에너지 위기 심화와 기업의 자산 오용 문제가 시장 전반의 하락을 주도했으나, 반도체 등 핵심 산업의 구조적 강세와 국내 일부 산업(프랜차이즈)의 활력 회복세가 혼조세를 보임."
}


In [22]:
final_prediction = {
    'generated_at': datetime.now().isoformat(),
    'kronos_predictions': predictions,
    'fused_analysis': fused
}

with open('../data/predictions.json', 'w', encoding='utf-8') as f:
    json.dump(final_prediction, f, ensure_ascii=False, indent=2)